# **Intro to autosklearn**

**AutoML**, or Automated Machine Learning refers to techniques for **automatically discovering the better models and set of hyperparameters** for these models for a given ML application.

`autosklearn` is the AutoML version of traditional `sklearn` package, that has recently been released.

We will address here just the coding details with `autosklearn`, for more information on AutoML definition, tools and application cases check:

- AutoML definition on Wikipedia: https://en.wikipedia.org/wiki/Automated_machine_learning
- AutoML.org website which gathers tons of resources and advances in autoML: https://www.automl.org/
- `autosklearn` github repository: https://automl.github.io/auto-sklearn/master/
- `autosklearn` tutorial: https://machinelearningmastery.com/auto-sklearn-for-automated-machine-learning-in-python/


### **Install autosklearn**

As for today (feb-2021) autosklearn does not come preinstalled into Colaboratory VM. Therefore first task is install it:

In [ ]:
# install dependences
!sudo apt-get install build-essential swig
# install autosklearn
!pip install auto-sklearn

Reading package lists... Done
Building dependency tree       
Reading state information... Done
build-essential is already the newest version (12.8ubuntu1.1).
The following package was automatically installed and is no longer required:
  libnvidia-common-510
Use 'sudo apt autoremove' to remove it.
Suggested packages:
  swig-doc swig-examples swig4.0-examples swig4.0-doc
The following NEW packages will be installed:
  swig swig4.0
0 upgraded, 2 newly installed, 0 to remove and 21 not upgraded.
Need to get 1,086 kB of archives.
After this operation, 5,413 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu focal/universe amd64 swig4.0 amd64 4.0.1-5build1 [1,081 kB]
Get:2 http://archive.ubuntu.com/ubuntu focal/universe amd64 swig all 4.0.1-5build1 [5,528 B]
Fetched 1,086 kB in 0s (3,303 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debc

In [ ]:
# AFTER installing the previous packages, your have to restart the
# execution environment BEFORE running any other code like this:

# print autosklearn version
import autosklearn
print('autosklearn: %s' % autosklearn.__version__)

autosklearn: 0.15.0


### **Import dataset**

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
# url = "https://archive.ics.uci.edu/ml/machine-learning-databases/iris/iris.data"
# names = ['sepal-length', 'sepal-width', 'petal-length', 'petal-width', 'Class']
# dataset = pd.read_csv(url, names=names)

# let's use in this case the iris dataset directly form sklearn examples package
from sklearn.datasets import load_iris

iris = load_iris()
print(iris.DESCR)



.. _iris_dataset:

Iris plants dataset
--------------------

**Data Set Characteristics:**

    :Number of Instances: 150 (50 in each of three classes)
    :Number of Attributes: 4 numeric, predictive attributes and the class
    :Attribute Information:
        - sepal length in cm
        - sepal width in cm
        - petal length in cm
        - petal width in cm
        - class:
                - Iris-Setosa
                - Iris-Versicolour
                - Iris-Virginica
                
    :Summary Statistics:

    ============== ==== ==== ======= ===== ====================
                    Min  Max   Mean    SD   Class Correlation
    ============== ==== ==== ======= ===== ====================
    sepal length:   4.3  7.9   5.84   0.83    0.7826
    sepal width:    2.0  4.4   3.05   0.43   -0.4194
    petal length:   1.0  6.9   3.76   1.76    0.9490  (high!)
    petal width:    0.1  2.5   1.20   0.76    0.9565  (high!)
    ============== ==== ==== ======= ===== ===========

In [ ]:
print("Feature names:")
print(iris.feature_names)
print("Data(X):")
print(iris.data[:3])
print("Target names:[0   1   2]")
print(iris.target_names)
print("Target(y):")
print(iris.target)

Feature names:
['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
Data(X):
[[5.1 3.5 1.4 0.2]
 [4.9 3.  1.4 0.2]
 [4.7 3.2 1.3 0.2]]
Target names:[0   1   2]
['setosa' 'versicolor' 'virginica']
Target(y):
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 2 2 2 2 2 2 2 2 2 2 2
 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2
 2 2]


In [ ]:

df_iris = pd.DataFrame(iris.data, columns=iris.feature_names)
df_iris['target'] = pd.DataFrame(iris.target, columns=['target'])

df_iris.head()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0


### **Preprocessing dataset**

Split dataset in features and labels:

In [ ]:
# store feature matrix in "X"
X = iris.data

# store target vector in "y"
y = iris.target

split both datasets, fetatures and labels, in training and testing sets:

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)

normalize dataset:

In [ ]:
from sklearn.preprocessing import StandardScaler

'''
Se entrena el StandardScaler (usando fit) únicamente con los datos de entrenamiento
para calcular parámetros estadísticos (como la media y la desviación estándar)
que luego se usan para transformar los datos.

Al aplicar transform tanto a los datos de entrenamiento como a los de prueba,
se garantiza que ambos conjuntos se escalen de la misma manera,
usando únicamente la información del entrenamiento.
'''

sc = StandardScaler()
sc.fit(X_train)

# normalize features dataset
X_train_std = sc.transform(X_train)
X_test_std = sc.transform(X_test)

### **Define and train autosklearn model**

Installing dask distributed seems to be a pre-requisite for running autosklearn, so, let's go with it:

In [ ]:
!pip install dask distributed --upgrade

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 12.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 934.8/934.8 KB 28.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.2/280.2 KB 21.0 MB/s eta 0:00:00
  Attempting uninstall: psutil
    Found existing installation: psutil 5.4.8
    Uninstalling psutil-5.4.8:
      Successfully uninstalled psutil-5.4.8
  Attempting uninstall: dask
    Found existing installation: dask 2022.2.1
    Uninstalling dask-2022.2.1:
      Successfully uninstalled dask-2022.2.1
  Attempting uninstall: distributed
    Found existing installation: distributed 2022.2.1
    Uninstalling distributed-2022.2.1:
      Successfully uninstalled distributed-2022.2.1


In [ ]:
from autosklearn.classification import AutoSklearnClassifier
# from autosklearn.regression import AutoSklearnRegressor # for regression tasks
#autosklearn prueba con multiples tipos modelos y multiples hyperparametros
model = AutoSklearnClassifier(ensemble_size=10, # size of the end ensemble (minimum is 1). Se queda con los 10 mejores modelos
                              time_left_for_this_task=120, #the number of seconds the process runs for. Tiene 120 segundos para probar el maximo de modelos
                              per_run_time_limit=30) # maximum seconds allocated per model. Cada modelo, tiene maximo 30 segundos

model.fit(X_train_std, y_train) # begin fitting the search model
print(model.sprint_statistics()) # print statistics for the search

In [ ]:
# with the small amount of time we have set, autosklearn was able to
# fit just 25 models. Let's see the configuration for each one:
print(model.show_models())

{26: {'model_id': 26, 'rank': 1, 'cost': 0.0, 'ensemble_weight': 1.0, 'data_preprocessor': <autosklearn.pipeline.components.data_preprocessing.DataPreprocessorChoice object at 0x7fe9eac602e0>, 'balancing': Balancing(random_state=1, strategy='weighting'), 'feature_preprocessor': <autosklearn.pipeline.components.feature_preprocessing.FeaturePreprocessorChoice object at 0x7fe9ead5e9a0>, 'classifier': <autosklearn.pipeline.components.classification.ClassifierChoice object at 0x7fe9ead5ec10>, 'sklearn_classifier': LinearDiscriminantAnalysis(shrinkage=0.10875504258712398, solver='lsqr',
                           tol=7.102852089811469e-05)}}


In [ ]:
# check the fitting time and scores for each model, between many other data:
model.cv_results_

{'mean_test_score': array([0.95 , 0.95 , 0.875, 0.95 , 0.875, 0.95 , 0.9  , 0.95 , 0.875,
        0.975, 0.85 , 0.9  , 0.95 , 0.9  , 0.95 , 0.9  , 0.925, 0.925,
        0.975, 0.95 , 0.925, 0.925, 0.95 , 0.95 , 1.   , 0.7  , 0.975,
        0.85 , 0.9  , 0.   , 0.85 , 0.925, 0.925, 1.   , 0.975, 0.625,
        0.925, 0.825, 0.8  , 0.925, 0.975, 0.75 , 1.   , 0.9  ]),
 'rank_test_scores': array([ 9,  9, 33,  9, 33,  9, 27,  9, 33,  4, 36, 27,  9, 27,  9, 27, 19,
        19,  4,  9, 19, 19,  9,  9,  1, 42,  4, 36, 27, 44, 36, 19, 19,  1,
         4, 43, 19, 39, 40, 19,  4, 41,  1, 27]),
 'mean_fit_time': array([1.65409803, 1.76104021, 0.97482538, 1.64510751, 2.51584864,
        2.75023723, 1.55823398, 0.82446218, 1.51081038, 0.95604658,
        0.95263433, 2.16118312, 2.26553464, 2.62326288, 1.68868828,
        1.27660584, 1.48840475, 1.95930362, 2.41808963, 4.22588491,
        1.08490443, 1.73817563, 1.66601133, 1.96957612, 2.67032647,
        1.33264971, 0.87965369, 1.09115863, 1.847681

### **Evaluate the performance of the best model/ensemble**

In [ ]:
# in our case (having specified ensemble_size=10) the predictions
# will use an ensemble (basically the mean) of the 10 best models predictions
y_predictions = model.predict(X_test_std) # get predictions from the model

In [ ]:
y_predictions

array([0, 1, 1, 0, 2, 1, 2, 0, 0, 2, 1, 0, 2, 1, 1, 0, 1, 1, 0, 0, 1, 1,
       1, 0, 2, 1, 0, 0, 1, 2])

In [ ]:
# apart from the basic accuracy score we can compute more complex and significative score parameters:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
print(confusion_matrix(y_test, y_predictions))
print(classification_report(y_test, y_predictions))
print("General Accuracy score:", accuracy_score(y_test, y_predictions))

[[11  0  0]
 [ 0 13  0]
 [ 0  0  6]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        11
           1       1.00      1.00      1.00        13
           2       1.00      1.00      1.00         6

    accuracy                           1.00        30
   macro avg       1.00      1.00      1.00        30
weighted avg       1.00      1.00      1.00        30

General Accuracy score: 1.0
